### RAG Pipelines- Data Ingestion to Vector DB Piepline

In [6]:
import os

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:
## Read all the PDF's inside the directory
def process_all_mds(md_directory):
    """Process all MD Files in a directory"""
    all_documents = []
    md_dir = Path(md_directory)

    ## Find all MD Files recursively
    md_files = list(md_dir.glob("**/*.md"))

    print(f"Found {len(md_files)} MD Files to process")

    for md_file in md_files:
        print(f"\nProcessing {md_file.name}")
        try:
            loader = TextLoader(str(md_file))
            documents = loader.load()

            ## Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = md_file.name
                doc.metadata['file_type'] = 'md'

            all_documents.extend(documents)
            print(f"   Loaded {len(documents)} pages")

            # # Split the documents into smaller chunks
            # text_splitter = RecursiveCharacterTextSplitter(
            #     chunk_size=1000,
            #     chunk_overlap=200,
            #     length_function=len
            # )
            # split_documents = text_splitter.split_documents(documents)
            #
            # all_documents.extend(split_documents)
        except Exception as e:
            print(f"   Error processing: {e}")

    print(f"\nTotal MD Documents loaded and Processed: {len(all_documents)}")
    return all_documents

# Process all MD files in the data directory
all_md_documents = process_all_mds("../data")


Found 17 MD Files to process

Processing 00-overview.md
   Loaded 1 pages

Processing 02-request-flows.md
   Loaded 1 pages

Processing 05-integrations.md
   Loaded 1 pages

Processing 10-troubleshooting.md
   Loaded 1 pages

Processing 04-business-logic.md
   Loaded 1 pages

Processing 07-kafka-events.md
   Loaded 1 pages

Processing 06-database.md
   Loaded 1 pages

Processing 01-architecture.md
   Loaded 1 pages

Processing 08-error-handling.md
   Loaded 1 pages

Processing 11-glossary.md
   Loaded 1 pages

Processing rag-index-map.md
   Loaded 1 pages

Processing 03-api-contracts.md
   Loaded 1 pages

Processing 09-configuration.md
   Loaded 1 pages

Processing perfios-flow.md
   Loaded 1 pages

Processing income-assessment-flow.md
   Loaded 1 pages

Processing cap-flow.md
   Loaded 1 pages

Processing zenith-flow.md
   Loaded 1 pages

Total MD Documents loaded and Processed: 17


In [13]:
# ## Print the no. of words in each MD files
#
# ## Read all the MD's inside the directory
# def process_all_mds(md_directory):
#     """Process all MD Files in a directory"""
#     all_documents = []
#     md_dir = Path(md_directory)
#
#     ## Find all MD Files recursively
#     md_files = list(md_dir.glob("**/*.md"))
#
#     print(f"Found {len(md_files)} MD Files to process")
#
#     for md_file in md_files:
#         print(f"\nProcessing {md_file.name}")
#         try:
#             # loader = TextLoader(str(md_file))
#             # 1) Load all markdown files
#             loader = DirectoryLoader(
#                 path="../data/md_files",
#                 glob="**/*.md",
#                 loader_cls=TextLoader,
#                 loader_kwargs={"encoding": "utf-8"},
#             )
#             documents = loader.load()
#
#             ## Add source information to metadata
#             # 2) Print text + word count in your flow
#             for i, doc in enumerate(documents, 1):
#                 doc.metadata['source_file'] = md_file.name
#                 doc.metadata['file_type'] = 'md'
#
#                 source = doc.metadata.get("source", f"doc_{i}")
#                 text = doc.page_content
#                 # print(f"\n=== {source} ===")
#                 # print(text[:1000])  # preview (remove [:1000] for full text)
#                 print("Word count:", len(text.split()))
#
#             all_documents.extend(documents)
#             print(f"   Loaded {len(documents)} pages")
#
#             # # Split the documents into smaller chunks
#             # text_splitter = RecursiveCharacterTextSplitter(
#             #     chunk_size=1000,
#             #     chunk_overlap=200,
#             #     length_function=len
#             # )
#             # split_documents = text_splitter.split_documents(documents)
#             #
#             # all_documents.extend(split_documents)
#         except Exception as e:
#             print(f"   Error processing: {e}")
#
#     print(f"\nTotal MD Documents loaded and Processed: {len(all_documents)}")
#     return all_documents
#
# # Process all MD files in the data directory
# all_md_documents = process_all_mds("../data")

Found 3 MD Files to process

Processing 00-overview.md
Word count: 2041
Word count: 2099
Word count: 2994
   Loaded 3 pages

Processing 02-request-flows.md
Word count: 2041
Word count: 2099
Word count: 2994
   Loaded 3 pages

Processing 01-architecture.md
Word count: 2041
Word count: 2099
Word count: 2994
   Loaded 3 pages

Total MD Documents loaded and Processed: 9


In [8]:
all_md_documents


[Document(metadata={'source': '../data/md_files/00-overview.md', 'source_file': '00-overview.md', 'file_type': 'md'}, page_content="# Income Assessment Knowledge Base\n\nThis folder is the context pack for **RAG-based developer assistance and onboarding** for the `income-assessment-service`.\n\nThe knowledge base is designed to help developers quickly understand the service's **business domain, architecture, request flows, APIs, integrations, persistence, events, configuration, and troubleshooting paths** without having to inspect the entire repository manually.\n\nThe primary focus is on **production behavior that can be established from the current repository code and configuration**.\n\n---\n\n## 1. Service Overview\n\n### Service\n\n`income-assessment-service`\n\n### Technology\n\n* Language: Kotlin\n* Framework: Spring WebFlux\n* Persistence: MongoDB\n* Messaging: Kafka\n* Configuration: Spring/application configuration and service-specific configuration files\n\n### Business Doma

In [3]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [9]:
chunks=split_documents(all_md_documents)
chunks

Split 17 documents into 604 chunks

 Example chunk:
Content: # Income Assessment Knowledge Base

This folder is the context pack for **RAG-based developer assistance and onboarding** for the `income-assessment-service`.

The knowledge base is designed to help d...
Metadata: {'source': '../data/md_files/00-overview.md', 'source_file': '00-overview.md', 'file_type': 'md'}


[Document(metadata={'source': '../data/md_files/00-overview.md', 'source_file': '00-overview.md', 'file_type': 'md'}, page_content="# Income Assessment Knowledge Base\n\nThis folder is the context pack for **RAG-based developer assistance and onboarding** for the `income-assessment-service`.\n\nThe knowledge base is designed to help developers quickly understand the service's **business domain, architecture, request flows, APIs, integrations, persistence, events, configuration, and troubleshooting paths** without having to inspect the entire repository manually.\n\nThe primary focus is on **production behavior that can be established from the current repository code and configuration**.\n\n---\n\n## 1. Service Overview\n\n### Service\n\n`income-assessment-service`\n\n### Technology\n\n* Language: Kotlin\n* Framework: Spring WebFlux\n* Persistence: MongoDB\n* Messaging: Kafka\n* Configuration: Spring/application configuration and service-specific configuration files\n\n### Business Doma